# Module 15 — Self-Supervised & Contrastive Learning for Proteins

## TL;DR — Plain English

**The core insight:** Labels are expensive. But the *data itself* can be supervision.

- **BERT / Masked LM:** Take a sentence, mask 15% of the words, train a model to predict them. No human labels — the original sentence *is* the label.
- **SimCLR:** Show a model two augmented views of the same image. Train it so both views produce the *same* embedding. Images paired with themselves.
- **ESM2:** Exactly BERT, but for proteins. Mask amino acids, predict them from evolutionary context across 250M sequences from UniProt.

**Why does this matter for structural biology?**

| Dataset | Size | Labels |
|---------|------|--------|
| PDB (experimental structures) | ~200K proteins | Full 3D structure |
| UniProt (sequences) | **250 million** proteins | None |
| Labeled function datasets | ~100K | Function / family |

SSL unlocks the 250M unlabeled sequences. ESM2 trains on all of them and learns the "grammar" of proteins — which positions co-evolve, what substitutions are tolerated, where helices vs sheets tend to form. Then you fine-tune ESM2 with 500 labeled examples and beat models trained from scratch on 50,000.

**After this notebook you can:**
- Implement masked LM (BERT-style) for protein sequences from scratch
- Implement SimCLR (NT-Xent loss) and BYOL contrastive learning
- Explain why BYOL works without negative pairs
- Load and fine-tune ESM2 embeddings for downstream tasks
- Ace SSL interview questions at computational biology ML teams / structural biology research labs

## Beginner Teaching Frame

**Who should fully work through this notebook:** students who already understand transformers and supervised fine-tuning.

**How to study it on a first pass:**
- begin with the data-scarcity argument, not the loss equations
- understand what pretraining buys you before implementing SimCLR or BYOL
- connect SSL to ESM2 and downstream transfer

**Common traps:** focusing on jargon instead of the core question: how do we learn from unlabeled protein sequences?


## Canonical Resource Upgrade

Use these for orientation:
- [Meta ESM repository](https://github.com/facebookresearch/esm)
- [Hugging Face ESM model pages](https://huggingface.co/facebook/esm2_t6_8M_UR50D)
- [SimCLR paper](https://arxiv.org/abs/2002.05709) and [BYOL paper](https://arxiv.org/abs/2006.07733) after the big picture is clear


## Cross-References

### Prerequisites
- **00/06 — PyTorch Fundamentals:** Training loops, optimizers, `nn.Module` (required)
- **05/01 — Deep Learning & Fine-tuning:** Transformer architecture, attention, LoRA (required)
- **01/07 — HMMs & Sequence Models:** Profile HMMs, sequence probability models (helpful background)

### Related Modules
- **07/01 — AlphaFold3 Core:** Uses SSL-pretrained MSA features from ESM2 as input to Pairformer
- **10/01 — Fine-tuning (Capstone):** LoRA fine-tuning of ESM2 for ΔΔG prediction and TCR-pMHC binding

### What Comes Next
- **10/01 — LoRA Fine-tuning of ESM2:** Apply PEFT methods to adapt ESM2-650M for specific structural biology tasks with minimal GPU memory

### Industry Connection
ESM2 underpins protein design workflows at computational biology ML teams, EvolutionaryScale, and Meta FAIR. Understanding SSL is prerequisite to understanding why foundation models work for proteins.

In [ ]:
import torch
import torch.nn as nn

# Self-Supervised Learning: Masked Language Modeling (BERT-style)
class MaskedLM(nn.Module):
    """Masked Language Modeling pre-training for protein sequences.
    Input: masked tokens → predict original tokens at masked positions.
    Used in: ESM-2, ProtBERT, TAPE
    """
    def __init__(self, vocab_size=21, d_model=128, n_heads=4, n_layers=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads,
                                                   dim_feedforward=d_model*4,
                                                   batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, vocab_size)  # predict original token

    def forward(self, tokens, mask_positions=None):
        x = self.embed(tokens)
        x = self.encoder(x)
        return self.head(x)  # (B, L, vocab_size)

# Masking strategy: 15% tokens → 80% MASK / 10% random / 10% unchanged
def apply_mask(tokens, mask_token_id=20, vocab_size=20, mask_prob=0.15):
    labels = tokens.clone()
    mask = torch.rand(tokens.shape) < mask_prob
    tokens = tokens.clone()
    for idx in mask.nonzero():
        r = torch.rand(1)
        if r < 0.8:
            tokens[idx[0], idx[1]] = mask_token_id
        elif r < 0.9:
            tokens[idx[0], idx[1]] = torch.randint(0, vocab_size, (1,)).item()
    labels[~mask] = -100  # ignore non-masked positions in loss
    return tokens, labels, mask

torch.manual_seed(42)
model = MaskedLM(vocab_size=21, d_model=64, n_heads=4, n_layers=2)
B, L = 4, 32
tokens = torch.randint(0, 20, (B, L))
masked_tokens, labels, mask = apply_mask(tokens)
logits = model(masked_tokens)
loss = nn.CrossEntropyLoss(ignore_index=-100)(logits.view(-1, 21), labels.view(-1))
print(f"MLM logits: {logits.shape}")
print(f"Masked {mask.sum()} / {B*L} tokens ({mask.float().mean():.1%})")
print(f"MLM loss: {loss.item():.4f}")
print(f"ESM-2 uses this exact objective on UniRef50 (250M proteins)")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# SimCLR: Simple Contrastive Learning of Representations
class SimCLREncoder(nn.Module):
    """SimCLR: learn representations by contrasting augmented views.
    For proteins: augmentations = random subsequence, masking, mutation
    """
    def __init__(self, vocab_size=21, d_model=64, proj_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.Sequential(
            nn.TransformerEncoderLayer(d_model, 4, batch_first=True),
        )
        self.projector = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Linear(d_model, proj_dim)
        )

    def forward(self, x):
        h = self.embed(x)
        h = self.encoder(h)
        h_pool = h.mean(dim=1)  # mean pool
        z = self.projector(h_pool)
        return F.normalize(z, dim=-1)  # L2 normalize

def nt_xent_loss(z1, z2, temperature=0.5):
    """NT-Xent contrastive loss (InfoNCE)."""
    B = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)  # (2B, d)
    sim = z @ z.T / temperature      # (2B, 2B)
    # Mask self-similarity
    mask = torch.eye(2*B, dtype=bool)
    sim.masked_fill_(mask, -1e9)
    # Positive pairs: (i, i+B) and (i+B, i)
    targets = torch.cat([torch.arange(B, 2*B), torch.arange(B)])
    return nn.CrossEntropyLoss()(sim, targets)

torch.manual_seed(42)
model = SimCLREncoder()
B, L = 8, 24
seq1 = torch.randint(0, 20, (B, L))
seq2 = torch.randint(0, 20, (B, L))  # augmented view
z1, z2 = model(seq1), model(seq2)
loss = nt_xent_loss(z1, z2)
print(f"SimCLR: z1={z1.shape}, z2={z2.shape}")
print(f"NT-Xent loss: {loss.item():.4f}")
print("Lower loss → representations of same protein are closer than negatives")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

# BYOL: Bootstrap Your Own Latent
class BYOL(nn.Module):
    """BYOL: online network predicts target network representation.
    No negative pairs needed! Uses asymmetric architecture + momentum target.
    """
    def __init__(self, encoder, proj_dim=32, pred_dim=32):
        super().__init__()
        # Online network
        self.online = encoder
        self.online_proj = nn.Sequential(
            nn.Linear(proj_dim, proj_dim), nn.BatchNorm1d(proj_dim), nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
        self.predictor = nn.Sequential(
            nn.Linear(proj_dim, pred_dim), nn.BatchNorm1d(pred_dim), nn.ReLU(),
            nn.Linear(pred_dim, proj_dim)
        )
        # Target network (momentum copy, NOT trained by backprop)
        self.target = copy.deepcopy(encoder)
        for p in self.target.parameters():
            p.requires_grad = False

    def update_target(self, momentum=0.996):
        for p_online, p_target in zip(self.online.parameters(),
                                       self.target.parameters()):
            p_target.data = momentum * p_target.data + (1-momentum) * p_online.data

    def forward(self, x1, x2):
        z1_online = self.online_proj(self.online(x1))
        z2_online = self.online_proj(self.online(x2))
        p1 = self.predictor(z1_online)
        p2 = self.predictor(z2_online)
        with torch.no_grad():
            z1_target = self.online_proj(self.target(x1)).detach()
            z2_target = self.online_proj(self.target(x2)).detach()
        loss = 2 - 2*(F.normalize(p1)*F.normalize(z2_target)).sum(-1).mean() +                2 - 2*(F.normalize(p2)*F.normalize(z1_target)).sum(-1).mean()
        return loss / 2

torch.manual_seed(42)
# Reuse encoder from above
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32))
    def forward(self, x): return self.net(x.float())

encoder = SimpleEncoder()
byol = BYOL(encoder, proj_dim=32)
x1 = torch.randint(0, 20, (8, 20))
x2 = torch.randint(0, 20, (8, 20))
loss = byol(x1, x2)
print(f"BYOL loss: {loss.item():.4f}")
byol.update_target(momentum=0.996)
print("BYOL: no negative pairs needed → avoids mode collapse via asymmetry")
print("Used in: ProteinBERT contrastive variant, ESM improvements")

In [ ]:
import torch
import torch.nn as nn

# ESM2 architecture and downstream fine-tuning
print("ESM-2: Protein Language Model Architecture")
print("=" * 60)
print()
print("ESM-2 family:")
esm2_models = {
    "esm2_t6_8M":   {"layers": 6,  "d_model": 320,  "heads": 20, "params": "8M"},
    "esm2_t12_35M": {"layers": 12, "d_model": 480,  "heads": 20, "params": "35M"},
    "esm2_t30_150M":{"layers": 30, "d_model": 640,  "heads": 20, "params": "150M"},
    "esm2_t33_650M":{"layers": 33, "d_model": 1280, "heads": 20, "params": "650M"},
    "esm2_t36_3B":  {"layers": 36, "d_model": 2560, "heads": 40, "params": "3B"},
}
for name, cfg in esm2_models.items():
    print(f"  {name}: {cfg['layers']} layers, d={cfg['d_model']}, h={cfg['heads']}, {cfg['params']}")

print()
print("Downstream tasks and fine-tuning strategies:")
tasks = {
    "Contact prediction": "logistic regression on attention maps (linear probe)",
    "Secondary structure": "per-token classification (1 of H/E/C)",
    "Stability (ΔΔG)": "regression on [CLS] or mean-pool embeddings",
    "Function annotation": "multi-label classification",
    "Solubility": "binary classification",
    "Localization": "multi-class classification (nucleus, cytoplasm...)",
}
for task, approach in tasks.items():
    print(f"  {task}: {approach}")

# Simulate ESM-style forward pass
class ESM2Lite(nn.Module):
    def __init__(self, vocab_size=30, d_model=64, n_heads=4, n_layers=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model*4,
                                               batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

    def forward(self, tokens):
        return self.encoder(self.embed(tokens))

torch.manual_seed(42)
esm_lite = ESM2Lite(n_layers=2)
x = torch.randint(0, 30, (2, 50))
embeddings = esm_lite(x)
print(f"\nESM-lite output: {embeddings.shape}  (batch, seq_len, d_model)")
print(f"Parameters: {sum(p.numel() for p in esm_lite.parameters()):,}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Downstream fine-tuning: ESM2 + classification head
class ESM2FineTuned(nn.Module):
    """ESM2 backbone (frozen/LoRA) + task-specific head."""
    def __init__(self, d_model=128, n_classes=3, freeze_backbone=True):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Embedding(21, d_model),
            nn.TransformerEncoder(
                nn.TransformerEncoderLayer(d_model, 4, batch_first=True, norm_first=True),
                num_layers=4
            )
        )
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, n_classes)
        )

    def forward(self, tokens):
        h = self.backbone[1](self.backbone[0](tokens))
        pooled = h.mean(dim=1)
        return self.head(pooled)

torch.manual_seed(42)
# Task: predict protein localization (3 classes: nucleus, cytoplasm, secreted)
model = ESM2FineTuned(d_model=64, n_classes=3, freeze_backbone=True)
B, L = 8, 64
tokens = torch.randint(0, 21, (B, L))
logits = model(tokens)
probs = F.softmax(logits, dim=-1)
print(f"Input: {tokens.shape} → logits: {logits.shape}")
print(f"Predicted classes: {logits.argmax(1).tolist()}")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}/{total:,} ({trainable/total:.1%}) — head only")
print()
print("Fine-tuning workflow:")
print("  1. Load ESM-2 pretrained weights from Hugging Face")
print("  2. Add task head; freeze backbone")
print("  3. Train on labeled data (e.g. SwissProt annotations)")
print("  4. Optionally unfreeze last N layers or use LoRA")

In [ ]:
import torch
import numpy as np

# SSL comparison: MLM vs SimCLR vs BYOL
print("SSL Methods Comparison for Protein Representation Learning")
print("=" * 70)
print()

comparison = {
    "Method": ["MLM (BERT/ESM)", "SimCLR", "BYOL", "MAE (ESM-3 style)", "Contrastive + Struct"],
    "Negatives": ["No (generative)", "Yes (large batch)", "No (momentum)", "No (reconstructive)", "Yes (structure pairs)"],
    "Augmentations": ["Masking 15%", "Subseq + mutation", "Subseq + noise", "Masking 75%+", "Homologs"],
    "Memory": ["Low", "High (large batch)", "2× encoder", "Low", "Medium"],
    "Best for": ["Language tasks", "Clustering/retrieval", "Clustering/retrieval", "Rich generation", "Structure-function"],
    "Example": ["ESM-2, ProtBERT", "ProteinCLIP", "MoCo-protein", "ESM-3, DNABERT-2", "AlphaFold3 MSA"],
}

headers = comparison["Method"]
for key in ["Negatives", "Augmentations", "Memory", "Best for", "Example"]:
    print(f"{key:<25}", end="")
    for i, val in enumerate(comparison[key]):
        print(f"  {val:<22}", end="")
    print()

print()
print("General rule:")
print("  Small labeled data → MLM pretrain + linear probe")
print("  Need clusters/retrieval → SimCLR or BYOL")
print("  Need generation → MAE / MLM")
print("  Protein structure prediction → MLM (language) + structure objectives")

## Interview Q&A

**Q: What is the difference between self-supervised and unsupervised learning?**

A: Unsupervised learning finds structure in data without labels: clustering (k-means), dimensionality reduction (PCA), density estimation (GMMs). Self-supervised learning generates *pseudo-labels from the data itself* and trains a discriminative model: predict masked tokens (BERT/ESM2), predict augmented views (SimCLR/BYOL), predict next token (GPT). SSL learns rich, transferable representations suitable for fine-tuning; classical unsupervised methods typically don't generalize as well to downstream discriminative tasks.

---

**Q: Why does BYOL work without negative pairs? Doesn't it collapse to a constant?**

A: Three mechanisms prevent collapse: (1) **EMA target**: the target network lags behind online, so online must always "chase" a moving target — it can't collapse if the target keeps moving. (2) **Predictor asymmetry**: only the online network has a predictor, so the target's representations don't receive the gradient signal to collapse. (3) **BatchNorm implicit contrastive signal**: batch statistics ensure variance is maintained. Formal proof: the EMA update creates an implicit contrastive term. SimSiam (no EMA) relies purely on the predictor stop-gradient and shows that even simpler configurations avoid collapse.

---

**Q: Why is ESM2 better than random initialization for protein tasks?**

A: ESM2 has seen 250M protein sequences and learned: (1) the statistical grammar of proteins — which residue patterns are evolutionarily tolerated; (2) co-evolutionary signals that correlate with 3D contacts; (3) structural motifs (helix, sheet, loop signatures); (4) functional signatures (active site patterns, binding interfaces). Starting from ESM2, even a *linear* classifier can separate protein families, predict secondary structure (~75% Q3 accuracy), and predict contact maps. Random init needs thousands of labeled examples to learn these patterns from scratch.

---

**Q: What is the NT-Xent loss and why is temperature important?**

A: NT-Xent = Normalized Temperature-scaled Cross Entropy. Given 2B embeddings (B originals + B augmented), for each sample $i$, treat its augmented pair as the positive and all $2B-2$ others as negatives. The loss is cross-entropy over the softmax of cosine similarities scaled by $\tau$:
$$\mathcal{L}_i = -\log \frac{\exp(\text{sim}(z_i, z_{i+B})/\tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$
Temperature $\tau$ controls the sharpness: **low $\tau$** focuses on hard negatives (can cause gradient instability for false negatives); **high $\tau$** treats all negatives equally (poor learning). Optimal $\tau \approx 0.1$–$0.5$.

---

**Q: SimCLR requires large batches (4096+) — why?**

A: NT-Xent loss needs many negatives for a meaningful gradient signal. With batch_size=32, there are only 30 negatives per sample — they might all be from different classes, making the task trivially easy. With B=4096, there are 4094 negatives including hard ones (similar proteins from different families). BYOL, MoCo (momentum queue of 65K negatives), and SwAV (prototypes) all solve this without requiring giant batches.

---

**Q: Name the three families of self-supervised learning methods.**

A: (1) **Contrastive**: explicit negative pairs [SimCLR, MoCo, SupCon]. (2) **Non-contrastive / self-distillation**: no negatives, uses EMA/stop-gradient [BYOL, SimSiam, DINO, EMA-based methods]. (3) **Clustering-based**: online clustering assigns pseudo-labels [SwAV, PCL, SCAN]. For proteins: ESM2 is masked generative SSL (a 4th category — generative/reconstructive), which is closest to MAE (Masked Autoencoders).

## Time Complexity

| Method | Training (per step) | Inference | Memory |
|--------|---------------------|-----------|--------|
| Masked LM | $O(L^2 \cdot B \cdot d)$ attention | $O(L^2 \cdot d)$ | $O(L^2)$ attn matrix |
| SimCLR | $O(B^2 \cdot d)$ similarity matrix | $O(L \cdot d)$ | $O(B)$ for negatives |
| BYOL | $O(B \cdot d)$ | $O(L \cdot d)$ | $O(2B)$ dual networks |
| ESM2-650M | Days on 512 A100s | ~10ms/seq (GPU) | 2.5GB model weights |
| ESM2-8M | Hours on 1 GPU | ~2ms/seq | 30MB model weights |

## Resources

### 1. Theory (Foundational Papers)
- **SimCLR**: "A Simple Framework for Contrastive Learning" (Chen et al., 2020): https://arxiv.org/abs/2002.05709
- **BYOL**: "Bootstrap Your Own Latent" (Grill et al., 2020): https://arxiv.org/abs/2006.07733
- **MoCo v3** (He et al., 2021): https://arxiv.org/abs/2104.02057
- **SimSiam** (Chen & He, 2021) — no EMA, stop-gradient only: https://arxiv.org/abs/2011.10566
- **DINO** (Caron et al., 2021) — self-distillation for ViTs: https://arxiv.org/abs/2104.14294
- **ESM2**: "Language models of protein sequences at the scale of evolution" (Lin et al., 2022): https://www.science.org/doi/10.1126/science.ade2574
- **Lilian Weng SSL overview**: https://lilianweng.github.io/posts/2021-05-31-contrastive/
- **"Self-supervised learning: The dark matter of intelligence"** (LeCun/Misra, Meta AI blog): https://ai.facebook.com/blog/self-supervised-learning-the-dark-matter-of-intelligence/

### 2. Must-Have Popular Resources

**Start Here (Zero Background):**
- Yannic Kilcher "SimCLR explained" YouTube (30 min, excellent visuals)
- Lilian Weng blog "Contrastive SSL" — best single-page overview
- HuggingFace ESM2 tutorial: https://huggingface.co/blog/esm2
- "The Illustrated BERT" by Jay Alammar: https://jalammar.github.io/illustrated-bert/

**Intermediate/Advanced:**
- Meta ESM GitHub (all ESM models + tutorials): https://github.com/facebookresearch/esm
- Annotated Transformer (implementation walkthrough): https://nlp.seas.harvard.edu/annotated-transformer/
- ProteinBERT (bio-specific pre-training): https://github.com/nadavbra/protein_bert
- DNABERT (DNA sequences): https://github.com/jerryji1993/DNABERT

### 3. Practicals (Hands-On)
- HuggingFace ESM2-8M: https://huggingface.co/facebook/esm2_t6_8M_UR50D
- ESM2 fine-tuning tutorial: https://huggingface.co/blog/esm2
- Kaggle: "Protein Function Prediction" using ESM2 embeddings
- ProteinGym: benchmark for SSL model evaluation: https://github.com/OATML-Markslab/ProteinGym
- FLIP (Fitness Landscape Inference for Proteins): https://github.com/J-SNACKKB/FLIP

### 4. Real-World Problems to Solve
1. **Protein thermostability prediction with ESM2**: Use ESM2-650M embeddings + linear probe on ProteinGym delta-Tm data. Dataset: Mega-scale protein stability dataset (Tsuboyama et al., 2023). Metric: Spearman correlation.
2. **Antimicrobial peptide classification**: SimCLR pre-train on unlabeled peptide sequences (APD3 database), fine-tune on labeled AMP/non-AMP. Compare SSL vs supervised-only.
3. **DNA regulatory region prediction**: DNABERT / Nucleotide Transformer for promoter vs enhancer classification on ENCODE regulatory elements.

### 5. Interview Tips for computational biology ML teams / structural biology research labs
- Know the **3 families**: (1) Contrastive [SimCLR, MoCo], (2) Non-contrastive [BYOL, SimSiam], (3) Clustering-based [SwAV, DINO]
- **DINO** = self-distillation with no labels — used for ViT protein structure foundation models
- **ESM2 is mandatory knowledge** for any protein ML role at computational biology ML teams/structural biology research labs
- Be able to implement NT-Xent loss from scratch (common whiteboard question)
- Explain: why does BYOL work? (EMA + predictor asymmetry, implicit contrastive via BN)
- Explain: SimCLR needs large batches, BYOL doesn't — and why each strategy addresses it
- Key connection: masked autoencoding (MAE, ESM2) ~ contrastive SSL in representation quality
- Know LoRA for ESM2 fine-tuning: reduces trainable params from 650M to ~1M (module 10/01)

### 6. Hot Industry Topics (2024-2025)
- **ESM3 (2024)**: Generative multimodal model (sequence + structure + function), trained with masked generative modeling. Can design novel proteins.
- **DINO/DINOv2**: Foundation model distillation for zero-shot generalization — being applied to protein structure embeddings.
- **Prodigy/ProSST**: Structure-aware self-supervised models combining sequence + structure graphs.
- **Multi-modal SSL**: Joint training on sequence + structure + GO terms + protein-protein interaction networks.
- **AlphaFold3**: Uses ESM2 SSL-pretrained embeddings as initial sequence features fed into Pairformer.
- **Foundation models for genomics**: HyenaDNA (long-context), Nucleotide Transformer (BERT-style), Evo (autoregressive, 7B params, 1M context).
- **Protein fitness prediction**: ESM2 zero-shot log-likelihood predicts mutant fitness — no fine-tuning needed for some tasks.

## Timed Practice Problems

Work through these without looking at the notebook cells above. Time yourself.

---

### Problem 1 (10 min) — Implement NT-Xent from Scratch

Given two tensors `z1` and `z2` of shape `(B, d)` that are already L2-normalized:
- Implement `nt_xent_loss(z1, z2, temperature=0.5)` without using the version from this notebook
- Handle the self-similarity masking correctly
- Verify: when `z1 == z2` (trivial case), what happens to the loss?

**Hint:** Build the `(2B, 2B)` similarity matrix, mask the diagonal, then use `F.cross_entropy`.

---

### Problem 2 (2 min) — BERT-Style Masking Rules

Describe from memory the **3 rules for BERT-style masking**:
1. What fraction of tokens are selected for masking?
2. Of those selected, what happens to each? (Give the 80/10/10 breakdown with explanation)
3. Why is the 10% "unchanged" category needed? What would go wrong without it?

---

### Problem 3 (5 min) — Why BYOL Does Not Collapse

Explain in **exactly 3 sentences** why BYOL does not collapse to a constant representation:
- Sentence 1: The role of the EMA target network
- Sentence 2: The role of the predictor (and why having it only in the online network matters)
- Sentence 3: What "collapse" would look like and why these mechanisms prevent it

---

### Problem 4 (3 min) — Projector vs Predictor in BYOL

Answer these in writing:
- What is the projector? Which networks have it?
- What is the predictor? Which networks have it?
- What happens if you add a predictor to the target network too?
- At inference time, which output do you use for downstream tasks: the projector output, the predictor output, or the encoder output? Why?

---

### Problem 5 (15 min) — Design Question: ESM2 for Protein Solubility

You have:
- ESM2-650M (650M param, 1280-dim embeddings per residue)
- 500 labeled proteins with solubility scores (continuous, 0-100%)
- Budget: 1x A100 40GB GPU, 4 hours compute

Design a complete fine-tuning pipeline:
1. How do you convert per-residue ESM2 embeddings to a per-protein representation? (pooling strategy)
2. Linear probe vs LoRA fine-tuning — which do you choose given the constraints? Why?
3. What loss function? What metric for evaluation?
4. What data splits would you use with 500 examples?
5. What baselines would you compare against?
6. What failure modes should you monitor?

*(See module 10/01 for the LoRA implementation that solves exactly this problem.)*

## Mastery Check

On a first pass, success means you can:
1. explain masked language modeling in plain English
2. explain SimCLR vs BYOL at a high level
3. explain why unlabeled protein data is valuable
4. connect SSL pretraining to downstream protein tasks


---
## 🧬 ESM-2 Embeddings — What Do Protein Language Models Learn?

This section shows what ESM-2 embeddings actually look like in practice — the single most impactful visualization in protein ML.


In [ ]:
# ESM-2 EMBEDDING VISUALIZATION
# Show that ESM-2 embeddings cluster by biological function WITHOUT any structure labels

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Diverse proteins representing different structural/functional classes
proteins = {
    # Kinases (share ATP-binding scaffold)
    'PKA_catalytic':    'MGNAAAAKKNSDSSPSTQSTPAAPNIINEQGVSEDLERLKEDLFSGFNLEINLTAELLRLSQNVQNKIQETFEVELEEK',
    'CDK2':             'MENFQKVEKIGEGTYGVVYKARNKLTGEVVALKKIRLDTETEGVPSTAIREISLLKELRHPNIVKLLDVIHTENKLYLVFEFLHQDLKKFMDASALTGIPLPLIKSYLFQLLQGLAFCHSHRVLHRDLKPQNLLINTEGAIKLADFGLARAFGVPVRTYTHEVVTLWYRSPEVLLGSARYSTPVDIWSIGTIFAELATKKPLFHGDSEIDQLFRIFRTLGTPDEVVWPGVTSMPDYKPSFPKWARQDFSKVVPPLDEDGRSLLSQMLHYDPNKRISAKAALAHPFFQDVTKPVPHLRL',
    # Antibodies (immunoglobulin fold)
    'IgG_VH':           'QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYGISWVRQAPGQGLEWMGWISAYNGNTNYAQKLQGRVTMTTDTSTSTAYMELRSLRSDDTAVYYCAR',
    'IgG_VL':           'DIQMTQSPSSLSASVGDRVTITCRASQGIRNDLGWYQQKPGKAPKRLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQYNTFPYTFGQGTKVEIK',
    # Globins (oxygen binding)
    'HBB_human':        'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH',
    'Myoglobin':        'MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG',
    # Disordered (should cluster apart)
    'p53_TAD':          'MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPG',
    'p27_IDR':          'MSNNVPVNARATVKKFIESLEPADPSDPSISDTDMNPVHSQGKKTKTISRSRSLSPLAQLLVQSTFLGPLQRSRVKSGKKRDQLGGMESGGGKELDDVLGKGMSASQLPNSSSFPQPQGSRRGAVNSSRAAAASQAFGSRGVGQSHRSGCPTKLFGAPSSPGQQALCPPSTGSSPGPQQS',
    # Membrane proteins (TM helices)
    'GPCR_fragment':    'MGQPGNGSAFLLAPNRSHAPDHDVTQQRDEVWVVGMGIVMSLIVLAIVFGNVLVITAIAKFERLQTVTNYFITSLACADLVMGLAVVPFGAAHILMKMWTFGNFWCEFWTSIDVLCVTASIETLCVIAVDRYFAITSPFKYQSLLTKNKARVIILMVWIVSGLTSFLPIQMHWYRATHQEAINC',
}

# Try to get real ESM-2 embeddings
embeddings = None
try:
    import esm
    model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    model.eval()

    import torch
    data = [(name, seq[:200]) for name, seq in proteins.items()]  # truncate for speed
    _, _, tokens = batch_converter(data)

    with torch.no_grad():
        results = model(tokens, repr_layers=[6])
    embeddings = results['representations'][6][:, 1:-1, :].mean(1).numpy()
    print("ESM-2 embeddings extracted (real model)")

except ImportError:
    print("fair-esm not installed — using simulated embeddings")
    print("Install with: pip install fair-esm")

if embeddings is None:
    # Simulate: proteins in same class cluster together
    np.random.seed(42)
    class_centers = {
        'Kinase': np.random.randn(320),
        'Antibody': np.random.randn(320),
        'Globin': np.random.randn(320),
        'Disordered': np.random.randn(320),
        'GPCR': np.random.randn(320),
    }
    # Normalize centers to unit sphere
    for k in class_centers:
        class_centers[k] /= np.linalg.norm(class_centers[k])

    protein_classes = {
        'PKA_catalytic': 'Kinase', 'CDK2': 'Kinase',
        'IgG_VH': 'Antibody', 'IgG_VL': 'Antibody',
        'HBB_human': 'Globin', 'Myoglobin': 'Globin',
        'p53_TAD': 'Disordered', 'p27_IDR': 'Disordered',
        'GPCR_fragment': 'GPCR',
    }
    embeddings = np.array([
        class_centers[protein_classes[name]] + np.random.randn(320) * 0.15
        for name in proteins
    ])
    labels = [protein_classes[n] for n in proteins]
    print(f"Simulated embeddings: {embeddings.shape}")
else:
    protein_classes = {
        'PKA_catalytic': 'Kinase', 'CDK2': 'Kinase',
        'IgG_VH': 'Antibody', 'IgG_VL': 'Antibody',
        'HBB_human': 'Globin', 'Myoglobin': 'Globin',
        'p53_TAD': 'Disordered', 'p27_IDR': 'Disordered',
        'GPCR_fragment': 'GPCR',
    }
    labels = [protein_classes[n] for n in proteins]

# PCA visualization
pca = PCA(n_components=2)
coords_pca = pca.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {'Kinase': '#1f77b4', 'Antibody': '#d62728', 'Globin': '#2ca02c',
          'Disordered': '#ff7f0e', 'GPCR': '#9467bd'}

for ax, (coords, title) in zip(axes, [(coords_pca, 'PCA')]):
    unique_classes = list(set(labels))
    for cls in unique_classes:
        mask = [l == cls for l in labels]
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   s=150, color=colors[cls], label=cls, zorder=5)
    for i, name in enumerate(proteins):
        ax.annotate(name.replace('_', '
'), (coords[i,0], coords[i,1]),
                    textcoords='offset points', xytext=(5, 3), fontsize=7)
    ax.set_title(f'ESM-2 Embeddings ({title})
'
                 f'Colored by protein class — NO structure labels used in training',
                 fontsize=10)
    ax.set_xlabel(f'{title}1')
    ax.set_ylabel(f'{title}2')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('esm2_tsne.png', dpi=100)
plt.show()

print("KEY OBSERVATION:")
print("  ESM-2 was trained ONLY on protein sequences with MLM (masked language modeling)")
print("  It was never shown 3D structures, functional annotations, or class labels.")
print()
print("  Yet: proteins from the same structural/functional family cluster together!")
print("  Kinases cluster with kinases, antibodies with antibodies, etc.")
print()
print("  WHY: Sequence → structure → function. ESM-2 learns the sequence patterns")
print("  that are diagnostic of each fold (contact patterns from co-evolution,")
print("  motif patterns from functional conservation).")
print()
print("  This is the foundational insight behind all protein language models.")

---
## 🔗 Cross-Module Connections — Self-Supervised Learning as the Foundation

### SSL as the Pre-Training Step for Everything

Self-supervised pre-training on unlabeled protein sequences (UniRef90, 250M sequences) is what makes ESM-2 powerful. All downstream modules build on this foundation.

| Module | How SSL (ESM-2) Is Used | What Features Are Used |
|--------|------------------------|----------------------|
| **10/01** Fine-Tuning | ESM-2 embeddings as input features for ΔΔG prediction | Last-layer representations (Layer 33 of ESM-2 650M) |
| **07/01** AF3 | AF3's input embedder uses MSA + single sequence features; SSL embeddings can augment | MSA features replace some of ESM-2's co-evolutionary signal |
| **12/01** Diffusion | Protein language model guides diffusion (EvoDiff, DPLM) | Masked-token objective = discrete diffusion |
| **14/01** RL | Latent space of ESM-2 used for RL protein search | ESM-2 hidden states as state representation |
| **17/00** Capstone | Simulated ESM-2 embeddings used for EGFR mutation features | Replace with real ESM-2 for production |

### The Masked Language Modeling ↔ Discrete Diffusion Connection

This is one of the most important conceptual unifications in modern ML:

```
BERT Masked LM:    x_masked = mask(x_original, 15%)
                   objective: predict original tokens at masked positions

Discrete Diffusion: x_t = corrupt(x_0, noise_level_t)
                    objective: predict x_0 from x_t at each noise level t

KEY INSIGHT: BERT = discrete diffusion where t is always fixed at one noise level
             ESM-2 is implicitly a diffusion model trained at one specific noise level
             EvoDiff, DPLM extend this to all noise levels → proper generative models
```

### How to Choose: ESM-2 vs AF3 Embeddings

| Use Case | Use ESM-2 | Use AF3 |
|----------|-----------|---------|
| Mutation effect prediction | ✓ Single-sequence, fast | ✗ Expensive for many variants |
| Binding site detection | ~ Limited (no structure) | ✓ Spatial + structural context |
| Sequence classification | ✓ Strong sequence features | ~ Overkill |
| Generative design | ✓ Sequence-level design | ✓ Structure-level design |
| Disordered regions | ✓ Better for IDRs | ✗ Low confidence anyway |

### Interview Q&A

**Q:** Why does ESM-2 understand protein structure even though it was only trained on sequences?
> **A:** Co-evolutionary signals. Residues that are spatially close tend to co-evolve (correlated mutations across homologs). The self-attention weights in ESM-2 implicitly learn these correlations — which ARE the contact map. Studies show ESM-2 attention maps predict residue-residue contacts with accuracy close to explicit co-evolutionary methods like CCMpred. This is why MLM on sequences gives structural understanding "for free."

**Q:** When would you use SimCLR-style contrastive learning instead of masked language modeling for proteins?
> **A:** When you have defined augmentations that preserve biological meaning. MLM is a generative objective — it learns to fill in any position. Contrastive learning is discriminative — it learns what makes two sequences "similar." For tasks where similarity is well-defined (same protein family, same fold class), contrastive pre-training can be more data-efficient. For general-purpose representations (like ESM-2), MLM is better because it doesn't require defining similarity.


## 🐛 Debug Exercise — Contrastive Self-Supervised Learning (SimCLR)

Find and fix the **3 bugs** in the SimCLR contrastive training code below.

**Expected correct output:**
```
NT-Xent loss: finite value (not NaN or inf)
Negative pairs: should NOT include the other augmented view of the same sequence
Downstream evaluation: use ENCODER features (before projection head)
```

<details>
<summary>Show answer</summary>

**Bug 1 — Temperature too low:** `τ=0.01` makes `exp(sim/τ)` extremely large, causing
numerical overflow and NaN loss. Standard SimCLR uses `τ=0.07` to `τ=0.5`.
Fix: `temperature = 0.1` (or 0.07).

**Bug 2 — Augmented view of same sequence as negative:** For sequence i, its augmented view
i' should be the positive pair. All other sequences j ≠ i (and their augmented views j')
are negatives. Including i' as a negative contradicts the objective.
Fix: mask out the diagonal AND the N-offset diagonal when computing the denominator.

**Bug 3 — Projection head at test time:** SimCLR's projection head `g(·)` is only used
during training to compute the contrastive loss. For downstream tasks, use the encoder
representation `h = f(x)` directly — the projection head discards class-relevant information.
Fix: `features = encoder(X_test)` (no `projector`).
</details>


In [ ]:
# DEBUG EXERCISE — SimCLR contrastive SSL, find and fix 3 bugs
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

N = 16    # batch size (N sequences, each augmented twice -> 2N samples)
d_enc = 64   # encoder output dim
d_proj = 32  # projection head output dim

# Simulated embeddings: z[i] and z[i+N] are augmented views of same sequence
z = F.normalize(torch.randn(2 * N, d_proj), dim=1)

# NT-Xent (normalized temperature-scaled cross-entropy) loss
def nt_xent_loss(z, temperature):
    z_norm = F.normalize(z, dim=1)
    sim = torch.mm(z_norm, z_norm.T)   # [2N, 2N] cosine similarity matrix

    # BUG 1: temperature too low — exp(sim/0.01) overflows to inf -> NaN loss
    sim_scaled = sim / temperature     # temperature=0.01 is way too small

    # BUG 2: the mask should exclude BOTH the diagonal (self-similarity)
    # AND the positive pair diagonal (z[i] and z[i+N]).
    # The current mask only removes self-similarity, leaving z[i+N] as a "negative" for z[i].
    mask_self = torch.eye(2 * N, dtype=torch.bool)
    # Missing: also mask the positive pairs (the other augmented view)
    # Fix: mask_pos_off = torch.zeros(2*N,2*N,dtype=torch.bool)
    #      mask_pos_off[:N, N:] = torch.eye(N, dtype=torch.bool)
    #      mask_pos_off[N:, :N] = torch.eye(N, dtype=torch.bool)
    #      mask = mask_self | mask_pos_off

    # Positive pairs: (i, i+N) and (i+N, i)
    labels = torch.cat([torch.arange(N, 2*N), torch.arange(0, N)])

    # Denominator includes the positive pair as a "negative" — BUG 2
    logits = sim_scaled.masked_fill(mask_self, -1e9)
    loss = F.cross_entropy(logits, labels)
    return loss

# BUG 1 demonstration
loss_bad = nt_xent_loss(z, temperature=0.01)
loss_ok  = nt_xent_loss(z, temperature=0.1)
print(f"NT-Xent loss (tau=0.01, too small): {loss_bad.item():.4f}  — may be NaN or very large")
print(f"NT-Xent loss (tau=0.1, reasonable): {loss_ok.item():.4f}")

# BUG 3: projection head applied at test time
class Encoder(nn.Module):
    def forward(self, x): return F.normalize(x, dim=1)

class Projector(nn.Module):
    def __init__(self): super().__init__(); self.fc = nn.Linear(d_enc, d_proj)
    def forward(self, h): return F.normalize(self.fc(h), dim=1)

encoder   = Encoder()
projector = Projector()

X_test = torch.randn(10, d_enc)
h = encoder(X_test)   # encoder features (should be used for downstream tasks)

# BUG 3: using projection head output for downstream evaluation
# projection head discards class-discriminative information
features_wrong   = projector(h)     # BUG: use projection head at test time
features_correct = h                # correct: use encoder features directly

print(f"\nProjection head output dim: {features_wrong.shape[1]} (discards info, should NOT use for eval)")
print(f"Encoder output dim: {features_correct.shape[1]} (use THIS for downstream tasks)")
